 Installation des bibliothèques

In [ ]:
#Installation
!pip install -q ultralytics opencv-python-headless matplotlib

 Imports et chargement du modèle


In [ ]:
#Imports
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from google.colab import files
import os
# Chargement d'un modèle YOLO déjà entraîné (YOLOv8 nano)
model = YOLO("yolov8n.pt")
# téléchargé automatiquement la première fois print("Modèle chargé avec succès !")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


 Téléversement de la vidéo

In [ ]:
# Upload de la vidéo (MP4)
uploaded = files.upload()
# clique sur "Choisir un fichier"
video_path = list(uploaded.keys())[0]
print(f"Vidéo chargée : {video_path}")

Saving istockphoto-1388796730-640_adpp_is.mp4 to istockphoto-1388796730-640_adpp_is.mp4
Vidéo chargée : istockphoto-1388796730-640_adpp_is.mp4


 Fonction de détection + dessin des boîtes

In [ ]:
def detecter_et_annoter(video_path, conf_threshold=0.5, min_area=0,
                        output_name="video_annotée.mp4", skip_frames=1):
    """
    Détecte les PERSONNES dans une vidéo et produit une vidéo annotée.

    Paramètres :
      - conf_threshold : est mon seuil de confiance (0.25 à 0.8)
      - min_area       : est le filtre de taille ( ou levier d'amélioration) - on ignore les petites boîtes
      - skip_frames    : va traiter 1 image sur N
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Erreur : impossible d'ouvrir la vidéo")
        return

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_name, fourcc, fps, (width, height))

    frame_id = 0
    detections_count = 0
    errors_examples = []
    # pour noter des erreurs plus tard

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # On peut sauter des frames pour aller plus vite

        if frame_id % skip_frames == 0:
            # Détection uniquement de la classe "person" (id=0 dans COCO)
            results = model(frame, conf=conf_threshold, classes=[0], verbose=False)

        for r in results:
            boxes = r.boxes
            for box in boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                area = (x2 - x1) * (y2 - y1)

                # Levier d'amélioration : filtre de taille
                if area < min_area:
                    continue

                # Dessiner le rectangle et le score
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"person {conf:.2f}"
                cv2.putText(frame, label, (x1, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                detections_count += 1
        out.write(frame)
        frame_id += 1

    cap.release()
    out.release()
    print(f"Vidéo annotée sauvegardée : {output_name}")
    print(f"Nombre de détections (après filtre) : {detections_count}")
    return output_name

Test avec différents seuils de confiance

In [ ]:
# Test 1 : seuil bas (beaucoup de détections, plus de faux positifs)
print("=== Seuil 0.25 ===")
detecter_et_annoter(video_path, conf_threshold=0.25,
                     output_name="annotée_seuil_025.mp4", skip_frames=2)

# Test 2 : seuil moyen (recommandé)
print("\n=== Seuil 0.50 ===")
detecter_et_annoter(video_path, conf_threshold=0.50,
                     output_name="annotée_seuil_050.mp4", skip_frames=2)

# Test 3 : seuil haut (moins de détections, plus de faux négatifs)
print("\n=== Seuil 0.70 ===")
detecter_et_annoter(video_path, conf_threshold=0.70,
                                                   output_name="annotée_seuil_070.mp4", skip_frames=2)

=== Seuil 0.25 ===
Vidéo annotée sauvegardée : annotée_seuil_025.mp4
Nombre de détections (après filtre) : 1018

=== Seuil 0.50 ===
Vidéo annotée sauvegardée : annotée_seuil_050.mp4
Nombre de détections (après filtre) : 946

=== Seuil 0.70 ===
Vidéo annotée sauvegardée : annotée_seuil_070.mp4
Nombre de détections (après filtre) : 854


'annotée_seuil_070.mp4'

Application d'un levier d'amélioration

In [ ]:
# Avant : seuil 0.40 sans filtre de taille
print("=== AVANT (sans filtre de taille) ===")
detecter_et_annoter(video_path, conf_threshold=0.40, min_area=0,
                    output_name="avant_filtre.mp4", skip_frames=2)

# Après : même seuil + filtre de taille (ex. 3000 pixels²)
print("\n=== APRÈS (avec filtre min_area=3000) ===")
detecter_et_annoter(video_path, conf_threshold=0.40, min_area=3000,
                    output_name="après_filtre.mp4", skip_frames=2)

print("\nCompare les deux vidéos : le filtre enlève souvent les petites fausses détections.")

=== AVANT (sans filtre de taille) ===
Vidéo annotée sauvegardée : avant_filtre.mp4
Nombre de détections (après filtre) : 976

=== APRÈS (avec filtre min_area=3000) ===
Vidéo annotée sauvegardée : après_filtre.mp4
Nombre de détections (après filtre) : 793

Compare les deux vidéos : le filtre enlève souvent les petites fausses détections.


 Télécharger les vidéos produites

In [ ]:
# On peut Télécharger les vidéos annotées sur notre ordinateur
files.download("annotée_seuil_050.mp4")
# la principale files.download("avant_filtre.mp4")
files.download("après_filtre.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>